# 00 — Data Acquisition

**Project:** Corporate Climate Credibility Audit — power-sector POC

**Audience:** HBS Climate and Sustainability Impact Lab (Toffel, Serafeim, Tufano)

**Notebook role:** Single source of truth for *all* raw-data acquisition in this project. Every downstream notebook reads from `data/raw/` or `data/processed/` produced here.

---

## Why this notebook exists

The credibility framework this project builds — combining corporate **Talk** (pledges), regulator-verified **Walk** (EPA emissions), and independent **Verify** (CEMS continuous stack measurement + Climate TRACE remote-sensing-derived estimates) — needs five distinct public data sources stitched together by a stable, reproducible pipeline.

Doing the acquisition once-and-correctly in a single, idempotent notebook is crucial for achieving the stated goal of the project. The EPA Greenhouse Gas Reporting Program (GHGRP) is, as of the September 2025 EPA proposal, scheduled to be dismantled for 46 of 47 source categories by mid-2026. The 2011–2023 record we are downloading here is the most complete authoritative emissions panel that will ever exist for this era of U.S. industry. Future reviewers re-running this notebook should get bit-for-bit identical inputs (verified via SHA-256 manifest) and the same downstream conclusions.

## What this notebook produces, in order

1. **EPA GHGRP** — verify the 5 pre-positioned workbooks on disk; SHA-256 each; re-download from EPA if any are missing.

2. **Cohort identification** — derive the top-50 U.S. fossil-fuel power-generating parent companies by 2023 attributed Subpart-D emissions. Excludes `NO_PARENT_RECORD` orphan facilities and backfills from rank 51+. Writes `data/processed/cohort_top50.csv` and `power_facility_parent_attribution_2023.csv`.

3. **EPA AMPD CEMS** — pull facility-unit-annual continuous-stack emissions for 2011–2024 via EPA's `streaming-services/emissions/apportioned/annual` endpoint, one year per request with rate-limit backoff. Idempotent: each year saves to `data/raw/cems/emissions_annual_{year}.csv`.

4. **Climate TRACE** — currently a manual one-time download of the USA country package (~350 MB) from https://climatetrace.org/data to `data/raw/climate_trace/climate_trace_us_power.zip`; the cell unpacks it automatically.

5. **SBTi** — fetch the public companies + targets workbooks from the SBTi dashboard.

6. **SEC EDGAR 10-Ks** — pull the most recent 10-K for each cohort parent that is an SEC registrant; explicitly block 25 known non-registrants (cooperatives, public-power authorities, PE-owned holdcos, defunct registrants like GenOn) via a `DO_NOT_MATCH` set to prevent fuzzy-match false positives.

7. **Manifest** — write `data/processed/data_manifest.json` with relative path, size, SHA-256, modification time, and source URL (where known) for every acquired file.

## Reproducibility contract

- Every download cell is **idempotent**: if a file is already on disk it is skipped (no re-download).
- All credentials live in `.env` (gitignored). A reviewer copies `.env.example` to `.env`, fills in two values, and runs the notebook.
- The data manifest written at the end allows any future reviewer to verify their downloaded files match ours bit-for-bit.

## Credentials needed

| Source | Credential | Where to obtain | Loaded by |
|---|---|---|---|
| EPA CAMD CEMS | API key (free, instant) | https://www.epa.gov/power-sector/cam-api-portal#request-api-key | Env var `EPA_CAMD_API_KEY` in `.env` |
| SEC EDGAR | Polite `User-Agent` (name + email) per SEC fair-use policy | n/a — you choose | Env var `SEC_EDGAR_UA` in `.env` |

Both are loaded automatically by cell 02 via `python-dotenv`. The cell explicitly resolves `.env` against the project root so the notebook works regardless of which directory Jupyter was launched from.

## Engineering decisions encoded here

These choices look like engineering minutiae but each fixes a real failure mode we encountered:

- **CEMS endpoint** — `streaming-services/emissions/apportioned/annual`, not `camd-services/bulk-files`. Bulk-files serves raw ECMPS submissions (monitoring plans, QA tests) and has a 15-minute manifest timeout that returns a degraded sample on auth failure.
- **CEMS auth** — header `x-api-key`, not query-string `?api_key=...`. The query-string form silently returns a 1-entry sample rather than failing.
- **CEMS pull strategy** — one year per request with 2-second inter-call sleep and exponential backoff (30s → 60s → 120s → 240s) on HTTP 429. EPA's effective short-window rate limit is tighter than the documented 1,000/hr.
- **Climate TRACE** — manual one-time download. EPA's auto-download URLs are version-fragile; the project log records the user-supplied URL.
- **CIK matching for 10-Ks** — explicit `DO_NOT_MATCH` blocklist for 25 known non-registrants + tightened fuzzy cutoff (0.92) + first-token gate operating on punctuation-normalized titles. Without these guards the matcher returned false positives (e.g., `GENON ENERGY INC` → `MGE ENERGY INC`; `UNS ENERGY CORP` → `US ENERGY CORP`) that would have poisoned downstream NLP scoring.
- **Cohort orphan exclusion** — facilities whose parent record is missing from the 2023 parent workbook get a `NO_PARENT_RECORD` placeholder. We exclude these from cohort ranking and backfill from rank 51+ so the cohort is 50 real parent companies rather than 46 parents plus 4 orphaned retired-coal-plant placeholders.

Each cell that needs manual user action prints `[ACTION REQUIRED]` so it cannot be missed.

## 0. Environment

Imports, paths, helpers, and `.env` loading. Run this first; everything below assumes these globals exist.

The `.env` loader explicitly resolves three candidate paths (project root, current directory, parent directory) and falls back to `find_dotenv()` walk-up. This guards against the failure mode where `load_dotenv()` is called without a path and silently fails when Jupyter is launched from `notebooks/` instead of the project root. After running cell 02 you should see `.env loaded from: …/task/.env`; if you see `.env NOT found …`, neither `EPA_CAMD_API_KEY` nor `SEC_EDGAR_UA` will be available to subsequent cells.

In [1]:
# --- standard library ---
from __future__ import annotations
import hashlib
import io
import json
import os
import re
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

# --- third-party (install if missing) ---
import pandas as pd
import requests
from tqdm.auto import tqdm

# Try to load .env if python-dotenv is available; otherwise expect env vars set in shell.
# We explicitly point to the project root so this works regardless of the kernel cwd.
try:
    from dotenv import load_dotenv, find_dotenv  # type: ignore
    _PROJECT_ROOT_FOR_ENV = Path(
        "/Users/souvikmandal/Documents/S00_Career-development/"
        "20260318_HBS_Sen-Data-Scientist/task"
    )
    _env_candidates = [
        _PROJECT_ROOT_FOR_ENV / ".env",
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
    ]
    _loaded_from = None
    for _p in _env_candidates:
        if _p.exists():
            load_dotenv(_p, override=False)
            _loaded_from = _p
            break
    if _loaded_from is None:
        # last-ditch attempt: walk up from cwd
        _found = find_dotenv(usecwd=True)
        if _found:
            load_dotenv(_found, override=False)
            _loaded_from = Path(_found)
    print(f".env loaded from: {_loaded_from}" if _loaded_from else ".env NOT found in any candidate location")
except ImportError:
    print("[warn] python-dotenv not installed; relying on shell-exported env vars only.")
    print("       Install with:  pip install python-dotenv")

# --- project paths ---
PROJECT_ROOT = Path(
    "/Users/souvikmandal/Documents/S00_Career-development/"
    "20260318_HBS_Sen-Data-Scientist/task"
)
DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EXTERNAL  = PROJECT_ROOT / "data" / "external"

for d in (
    DATA_RAW / "epa_ghgrp",
    DATA_RAW / "cems",
    DATA_RAW / "climate_trace",
    DATA_RAW / "sbti",
    DATA_RAW / "sec_10k",
    DATA_PROCESSED,
    DATA_EXTERNAL,
):
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"DATA_RAW         = {DATA_RAW}")
print(f"DATA_PROCESSED   = {DATA_PROCESSED}")
print(f"Python           = {sys.version.split()[0]}")
print(f"pandas           = {pd.__version__}")

.env loaded from: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/.env
PROJECT_ROOT     = /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task
DATA_RAW         = /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/raw
DATA_PROCESSED   = /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/processed
Python           = 3.10.11
pandas           = 2.3.3


/Users/souvikmandal/Documents/S01_Learning/IBM_Machine-Learning-Professional-Certificate/ML_Venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- helpers ---
def sha256_of(path: Path, chunk: int = 1 << 20) -> str:
    """Return SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def http_download(
    url: str,
    out_path: Path,
    headers: Optional[dict] = None,
    timeout: int = 60,
    chunk: int = 1 << 16,
    overwrite: bool = False,
) -> Path:
    """Stream a URL to disk with a progress bar. Idempotent unless overwrite=True."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not overwrite:
        print(f"  [skip] {out_path.name} already on disk ({out_path.stat().st_size:,} bytes)")
        return out_path
    print(f"  [get ] {url}")
    with requests.get(url, headers=headers or {}, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        with open(out_path, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=out_path.name, leave=False
        ) as bar:
            for blob in r.iter_content(chunk_size=chunk):
                f.write(blob)
                bar.update(len(blob))
    print(f"  [ok  ] {out_path.name} ({out_path.stat().st_size:,} bytes)")
    return out_path


def safe_print_head(path: Path, n: int = 3) -> None:
    """Best-effort head for quick eyeballing of acquired files."""
    try:
        if path.suffix == ".csv":
            print(pd.read_csv(path, nrows=n).to_string())
        elif path.suffix in (".xlsx", ".xlsb", ".xls"):
            print(f"  (Excel file — open in Section 1/2 for structured read)")
        elif path.suffix == ".json":
            print(json.dumps(json.loads(path.read_text())[:n] if isinstance(json.loads(path.read_text()), list) else json.loads(path.read_text()), indent=2)[:600])
    except Exception as e:
        print(f"  (could not preview: {e})")

## 1. EPA GHGRP — verify on-disk workbooks

The five GHGRP workbooks listed below are pre-positioned in `data/raw/epa_ghgrp/`. We verify each exists, record its SHA-256, and re-download from EPA's public archive only if missing.

**Authoritative source pages:**
- https://www.epa.gov/ghgreporting/ghgrp-reported-data
- https://www.epa.gov/ghgreporting/data-sets

**Files used downstream:**

| File | Contents | Used by |
|---|---|---|
| `ghgp_data_by_year_2023.xlsx` | Wide-form facility panel — one row per facility, separate columns for each year's emissions and subparts, 2011–2023; multiple sheets by emitter type (Direct Point Emitters, Onshore Oil & Gas Prod., LDC, etc.) | Sections 2, Notebook 01 |
| `ghgp_data_2023.xlsx` | 2023-only snapshot with gas-by-gas breakdown (CO₂, CH₄, N₂O, fluorinated) | Notebook 02 (gas-mix validation) |
| `ghgp_data_parent_company.xlsb` | Facility → parent-company mapping with ownership percentage; one sheet per year 2010–2023 | Section 2, Notebook 01 |
| `emissions_by_unit_and_fuel_type_c_d_aa.xlsb` | Unit-level and fuel-level emissions detail (Subparts C, D, AA) | Notebook 01 Section 3 (fuel-mix decomposition) |
| `ghgrp_oris_power_plant_crosswalk_12_13_21.xlsx` | GHGRP facility ID → ORIS power-plant ID crosswalk | Notebook 02 (CEMS join bridge) |

Together these workbooks let us reconstruct the 13-year power-sector emissions panel for the locked top-50 cohort, decompose emissions by fuel type, and join to CEMS via the ORIS crosswalk.

In [3]:
# Map of expected GHGRP files. URLs are EPA's public bulk-download endpoints;
# if any move, the cell prints a clear [ACTION REQUIRED] line.
GHGRP_FILES = {
    "ghgp_data_by_year_2023.xlsx": (
        "https://www.epa.gov/system/files/other-files/2024-10/ghgp_data_by_year_2023.xlsx"
    ),
    "ghgp_data_2023.xlsx": (
        "https://www.epa.gov/system/files/other-files/2024-10/ghgp_data_2023.xlsx"
    ),
    "ghgp_data_parent_company.xlsb": (
        "https://www.epa.gov/system/files/other-files/2024-10/ghgp_data_parent_company.xlsb"
    ),
    "emissions_by_unit_and_fuel_type_c_d_aa.xlsb": (
        "https://www.epa.gov/system/files/other-files/2024-10/"
        "emissions_by_unit_and_fuel_type_c_d_aa.xlsb"
    ),
    "ghgrp_oris_power_plant_crosswalk_12_13_21.xlsx": (
        "https://www.epa.gov/system/files/documents/2022-01/"
        "ghgrp_oris_power_plant_crosswalk_12_13_21.xlsx"
    ),
}

GHGRP_DIR = DATA_RAW / "epa_ghgrp"
ghgrp_status = []

for fname, url in GHGRP_FILES.items():
    p = GHGRP_DIR / fname
    if p.exists():
        digest = sha256_of(p)
        size   = p.stat().st_size
        ghgrp_status.append(
            {"file": fname, "status": "present", "size": size, "sha256": digest, "url": url}
        )
        print(f"[present] {fname:55s}  {size:>12,} B  sha256={digest[:12]}…")
    else:
        print(f"[missing] {fname:55s}  attempting download …")
        try:
            http_download(url, p)
            ghgrp_status.append(
                {"file": fname, "status": "downloaded", "size": p.stat().st_size,
                 "sha256": sha256_of(p), "url": url}
            )
        except Exception as e:
            print(f"  [ACTION REQUIRED] download failed: {e}")
            print(f"  Open in browser: https://www.epa.gov/ghgreporting/data-sets")
            print(f"  Save the file as: {p}")
            ghgrp_status.append(
                {"file": fname, "status": "MISSING", "size": None, "sha256": None, "url": url}
            )

pd.DataFrame(ghgrp_status)[["file", "status", "size", "sha256"]]

[present] ghgp_data_by_year_2023.xlsx                                 3,182,304 B  sha256=8cc575b5609b…
[present] ghgp_data_2023.xlsx                                         2,610,840 B  sha256=e348830722f5…
[present] ghgp_data_parent_company.xlsb                               7,984,329 B  sha256=5613dd8454b0…
[present] emissions_by_unit_and_fuel_type_c_d_aa.xlsb                11,690,938 B  sha256=6fa39cd5ac4b…
[present] ghgrp_oris_power_plant_crosswalk_12_13_21.xlsx                272,305 B  sha256=f4ec8ff08d3d…


,file,status,size,sha256
0,ghgp_data_by_year_2023.xlsx,present,3182304,8cc575b5609bc361ba6ca335beb1939f1403a70e096b99...
1,ghgp_data_2023.xlsx,present,2610840,e348830722f59de38640b6f3ddfc9b7d406f26c3c153ff...
2,ghgp_data_parent_company.xlsb,present,7984329,5613dd8454b08deb160e3f1dec18c1a9b45464bc0ef845...
3,emissions_by_unit_and_fuel_type_c_d_aa.xlsb,present,11690938,6fa39cd5ac4bc17560c64ccd93442c744c4d24945ff0c5...
4,ghgrp_oris_power_plant_crosswalk_12_13_21.xlsx,present,272305,f4ec8ff08d3d964f8531a011705dde1b228266d39c3f73...


## 2. Cohort identification — top-50 power-sector parents (2023 attributed emissions)

This section locks the analytical universe. The cohort is derived deterministically from GHGRP itself so it is replicable and contains no judgment calls.

**Construction logic:**

1. From `ghgp_data_by_year_2023.xlsx`, load the **Direct Point Emitters** sheet. This is the wide-form sheet containing one row per facility with 13 per-year emission columns and 13 per-year subpart columns plus stable metadata (lat/lon, state, etc.).
2. **Filter to Subpart D (Electricity Generation)** for the 2023 reporting year. The regex `(?<![A-Z])D(?![A-Z])` matches the bare letter `D` without falsely matching multi-letter codes.
3. From `ghgp_data_parent_company.xlsb`, load the **2023 sheet** and attach each facility's parent company plus ownership share. Multi-parent facilities (joint ownership) keep separate rows with shares that sum to ~1.0.
4. **Attribute** each facility's 2023 CO₂e by ownership share: a facility owned 60/40 contributes 60% of its tons to one parent and 40% to the other.
5. Roll up to parent level and rank by 2023 attributed emissions.
6. **Exclude `NO_PARENT_RECORD` orphan facilities** — these are retired plants (e.g., Navajo Generating Station, Bruce Mansfield, San Juan, J M Stuart) whose parent record is missing from the 2023 parent workbook. They are not parent companies; ranking them as such would inflate the cohort total share and misattribute emissions. We backfill from rank 51+ to maintain n=50.
7. Save the locked cohort to `data/processed/cohort_top50.csv` and the underlying facility-level attribution to `power_facility_parent_attribution_2023.csv`. Every downstream notebook reads these.

**Defensive parsing:** EPA's column names are year-prefixed (`2023 Total reported direct emissions`, `2023 Subparts`) rather than plain (`Total reported direct emissions`, `Subparts`). Both the cohort cell and the parent-workbook cell use regex-based inference fallbacks to handle these year-prefixed variants and the upper-case spellings (`PARENT COMPANY NAME`, `GHGRP FACILITY ID`) used in the parent workbook.

**Locked decision:** per scope discussion on 2026-05-22, ownership is fixed at the **2023** snapshot and applied across the entire 2011–2023 panel. M&A timeline reconstruction is a Phase-2 item, acknowledged as a limitation in the working paper.

In [4]:
# --- load 2023 facility-level GHGRP panel ---
ghgrp_year_xlsx = GHGRP_DIR / "ghgp_data_by_year_2023.xlsx"
print(f"Reading {ghgrp_year_xlsx.name} …")
# Inspect available sheets first so we pick the right one regardless of EPA's naming.
xl = pd.ExcelFile(ghgrp_year_xlsx)
print("Sheets available:")
for s in xl.sheet_names:
    print(f"  - {s}")

Reading ghgp_data_by_year_2023.xlsx …
Sheets available:
  - Direct Point Emitters
  - Onshore Oil & Gas Prod.
  - Gathering & Boosting
  - Transmission Pipelines
  - LDC - Direct Emissions
  - SF6 from Elec. Equip.
  - Suppliers
  - CO2 Injection
  - Geologic Sequestration of CO2
  - Industry Type
  - FAQs about this Data


In [5]:
# The annual-snapshot sheets in this workbook are named by year ("2023", "2022", …).
# Direct emitters sheet name typically: "Direct Emitters" or "2023". We probe both.
candidate_sheets = [s for s in xl.sheet_names if "2023" in s or "Direct" in s]
print("Candidate sheets:", candidate_sheets)

# Load the most likely facility-level 2023 sheet. EPA convention: header is in row 3 (0-indexed).
# We try a few header positions if needed.
def load_ghgrp_facility_year(xl_file: pd.ExcelFile, sheet: str) -> pd.DataFrame:
    for header_row in (3, 0, 1, 2, 4):
        try:
            df = pd.read_excel(xl_file, sheet_name=sheet, header=header_row)
            # Heuristic: must contain a recognizable facility id column.
            cols_lower = [str(c).lower() for c in df.columns]
            if any("facility id" in c or "ghgrp id" in c for c in cols_lower):
                print(f"  loaded sheet={sheet!r} with header={header_row}, shape={df.shape}")
                return df
        except Exception:
            continue
    raise RuntimeError(f"Could not parse sheet {sheet!r}")

# Try the canonical "Direct Emitters" sheet first; fall back to first candidate.
sheet_to_use = "Direct Emitters" if "Direct Emitters" in xl.sheet_names else candidate_sheets[0]
facilities_2023 = load_ghgrp_facility_year(xl, sheet_to_use)
facilities_2023.columns = [str(c).strip() for c in facilities_2023.columns]
facilities_2023.head(3)

Candidate sheets: ['Direct Point Emitters', 'LDC - Direct Emissions']
  loaded sheet='Direct Point Emitters' with header=3, shape=(8737, 26)


,Facility Id,FRS Id,Facility Name,City,State,Zip Code,Address,County,Latitude,Longitude,...,2020 Total reported direct emissions,2019 Total reported direct emissions,2018 Total reported direct emissions,2017 Total reported direct emissions,2016 Total reported direct emissions,2015 Total reported direct emissions,2014 Total reported direct emissions,2013 Total reported direct emissions,2012 Total reported direct emissions,2011 Total reported direct emissions
0,1004377,1.100438e+11,121 REGIONAL DISPOSAL FACILITY,MELISSA,TX,75454,3820 SAM RAYBURN HIGHWAY,COLLIN COUNTY,33.298570,-96.535860,...,504064.0,518680.25,653854.00,250497.50,221014.75,199011.75,241883.50,289953.25,204000.00,194000.0
1,1010040,1.100712e+11,15-18565/15-18662,Hazard,KY,40701,1021 Tori Drive,PERRY COUNTY,37.274127,-83.239034,...,144097.5,NaN,125981.75,218699.25,141439.50,118204.50,225708.25,306680.75,348450.75,390393.5
2,1010085,1.100555e+11,15-19015,Hazard,KY,41701,1845 S. KY HWY 15,PERRY COUNTY,37.236617,-83.181260,...,NaN,NaN,93918.75,70265.00,35907.00,55872.50,68761.00,57767.00,41513.00,64664.5


In [6]:
# Normalize column names we'll need downstream. The exact strings shift slightly between
# EPA releases — this defensive renamer is intentionally verbose.
COL_ALIASES = {
    "facility_id":     ["Facility Id", "GHGRP ID", "Facility ID", "Facility Identifier"],
    "facility_name":   ["Facility Name"],
    "state":           ["State"],
    "city":            ["City"],
    "county":          ["County"],
    "lat":             ["Latitude"],
    "lon":             ["Longitude"],
    "subparts":        ["Subparts", "Subpart", "Reported Subparts"],
    "co2e_tons":       [
        "Total reported direct emissions",
        "GHG QUANTITY (METRIC TONS CO2e)",
        "Total Reported Direct Emissions (Metric Tons CO2e)",
    ],
}

def rename_first_match(df: pd.DataFrame, aliases: dict) -> pd.DataFrame:
    rename = {}
    for canonical, candidates in aliases.items():
        for c in candidates:
            if c in df.columns:
                rename[c] = canonical
                break
    return df.rename(columns=rename)

facilities_2023 = rename_first_match(facilities_2023, COL_ALIASES)
print("Resolved columns:", [c for c in COL_ALIASES if c in facilities_2023.columns])
facilities_2023[[c for c in COL_ALIASES if c in facilities_2023.columns]].head(3)

Resolved columns: ['facility_id', 'facility_name', 'state', 'city', 'county', 'lat', 'lon']


,facility_id,facility_name,state,city,county,lat,lon
0,1004377,121 REGIONAL DISPOSAL FACILITY,TX,MELISSA,COLLIN COUNTY,33.298570,-96.535860
1,1010040,15-18565/15-18662,KY,Hazard,PERRY COUNTY,37.274127,-83.239034
2,1010085,15-19015,KY,Hazard,PERRY COUNTY,37.236617,-83.181260


In [7]:
# --- filter to Subpart D (Electricity Generation) ---
# The "Subparts" column is a comma- or space-separated string of subpart letters.
if "subparts" not in facilities_2023.columns:
    subparts_guess = next(
        (c for c in facilities_2023.columns if re.search(r"subpart", str(c), flags=re.IGNORECASE)),
        None,
    )
    if subparts_guess:
        facilities_2023 = facilities_2023.rename(columns={subparts_guess: "subparts"})
        print(f"[info] inferred subparts from column: {subparts_guess}")
    else:
        print("[warn] no subparts column found; falling back to no Subpart-D filter for this run.")
        facilities_2023["subparts"] = "UNKNOWN_NO_SUBPARTS_COLUMN"

# Some EPA vintages rename the emissions column; infer a usable numeric source if needed.
if "co2e_tons" not in facilities_2023.columns:
    emission_candidates = [
        c for c in facilities_2023.columns
        if re.search(r"co2e|co2.?e|direct emissions|metric tons|ghg quantity", str(c), flags=re.IGNORECASE)
    ]
    if emission_candidates:
        preferred_2023 = [c for c in emission_candidates if re.search(r"2023", str(c))]
        pool = preferred_2023 or emission_candidates
        best_col = max(
            pool,
            key=lambda c: pd.to_numeric(facilities_2023[c], errors="coerce").notna().sum(),
        )
        facilities_2023["co2e_tons"] = pd.to_numeric(facilities_2023[best_col], errors="coerce")
        print(f"[info] inferred co2e_tons from column: {best_col}")
    else:
        raise RuntimeError(
            "Could not find an emissions column to map to 'co2e_tons'. "
            "Review facility sheet columns and extend COL_ALIASES['co2e_tons']."
        )

facilities_2023["subparts"] = facilities_2023["subparts"].fillna("").astype(str)
if (facilities_2023["subparts"] == "UNKNOWN_NO_SUBPARTS_COLUMN").all():
    power_mask = pd.Series(True, index=facilities_2023.index)
else:
    power_mask = facilities_2023["subparts"].str.contains(r"(?<![A-Z])D(?![A-Z])", regex=True)

power_facilities = facilities_2023.loc[power_mask].copy()
print(f"Power-sector facilities (Subpart D in 2023): {len(power_facilities):,}")
print(f"Total 2023 facilities: {len(facilities_2023):,}")
print(f"Coverage: {len(power_facilities) / max(len(facilities_2023), 1):.1%}")

preview_cols = ["facility_id", "facility_name", "state", "subparts", "co2e_tons"]
preview_cols = [c for c in preview_cols if c in power_facilities.columns]
power_facilities[preview_cols].head(5)

[info] inferred subparts from column: Latest Reported Industry Type (subparts)
[info] inferred co2e_tons from column: 2023 Total reported direct emissions
Power-sector facilities (Subpart D in 2023): 1,390
Total 2023 facilities: 8,737
Coverage: 15.9%


,facility_id,facility_name,state,subparts,co2e_tons
5,1000112,23rd and 3rd,NY,"C,D",31916.132
25,1001106,48th Street Peaking Station,MI,"C,D",39263.556
31,1001033,A B Brown Generating Station,IN,D,1982200.776
76,1001444,AES Alamitos,CA,"C,D",1519041.820
77,1001222,AES Beaver Valley LLC,PA,D,NaN


In [8]:
# --- attach parent-company ownership (2023 sheet of parent workbook) ---
parent_xlsb = GHGRP_DIR / "ghgp_data_parent_company.xlsb"
print(f"Reading {parent_xlsb.name} …  (xlsb backend: pyxlsb)")

# pyxlsb is required to read .xlsb. Install on the fly if missing.
try:
    import pyxlsb  # noqa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyxlsb'])
    import pyxlsb  # noqa

parent_xl = pd.ExcelFile(parent_xlsb, engine="pyxlsb")
print("Parent workbook sheets:", parent_xl.sheet_names[:5], "…")

# Most-recent-ownership rule: use the 2023 sheet for the full panel attribution.
sheet_2023 = next((s for s in parent_xl.sheet_names if "2023" in str(s)), None)
if sheet_2023 is None:
    raise RuntimeError("No 2023 sheet found in parent_company workbook.")
parents_2023 = pd.read_excel(parent_xlsb, sheet_name=sheet_2023, engine="pyxlsb", header=0)
parents_2023.columns = [str(c).strip() for c in parents_2023.columns]
print(f"parents_2023.shape = {parents_2023.shape}")
parents_2023.head(3)

Reading ghgp_data_parent_company.xlsb …  (xlsb backend: pyxlsb)
Parent workbook sheets: ['2023', '2022', '2021', '2020', '2019'] …
parents_2023.shape = (9060, 16)


,GHGRP FACILITY ID,FRS ID (FACILITY),REPORTING YEAR,FACILITY NAME,FACILITY ADDRESS,FACILITY CITY,FACILITY STATE,FACILITY ZIP,FACILITY COUNTY,PARENT COMPANY NAME,PARENT CO. STREET ADDRESS,PARENT CO. CITY,PARENT CO. STATE,PARENT CO. ZIP,PARENT CO. PERCENT OWNERSHIP,FACILITY NAICS CODE
0,1000001,1.100005e+11,2023,PSE Ferndale Generating Station,5105 LAKE TERRELL ROAD,FERNDALE,WA,98248,WHATCOM COUNTY,PUGET HOLDINGS LLC,Po Box 97034,Bellevue,WA,98009,100.0,221112.0
1,1000002,1.100412e+11,2023,Ardagh Glass Inc. (Dunkirk),524 E. CENTER STREET,DUNKIRK,IN,47336,JAY COUNTY,ARDAGH GLASS INC,"10194 Crosspoint Blvd, Suite 410",Indianapolis,IN,46256,100.0,327213.0
2,1000003,1.100015e+11,2023,Ardagh Glass Inc. (Henderson),620 Facet Road,Henderson,NC,27537,VANCE COUNTY,ARDAGH GLASS INC,10194 CROSSPOINT BLVD STE 410,Indianapolis,IN,46256,100.0,327213.0


In [9]:
# Defensive renaming for parent-company sheet — column names shift across vintages.
PARENT_ALIASES = {
    "facility_id": [
        "GHGRP FACILITY ID", "GHGRP Facility ID", "Facility Id", "Facility ID",
        "Facility Identifier", "GHGRP ID",
    ],
    "parent_name": [
        "PARENT COMPANY", "Parent Company", "PARENT COMPANIES",
        "Parent Companies", "Parent Company Name", "Parent Name",
    ],
    "ownership_pct": [
        "PARENT CO. PERCENT OWNERSHIP", "Parent Company Ownership Percent",
        "PARENT CO PERCENT OWNERSHIP", "PERCENT OWNERSHIP", "Ownership Percent",
    ],
}
parents_2023 = rename_first_match(parents_2023, PARENT_ALIASES)

# Fallback inference if aliases miss due EPA schema drift.
if "parent_name" not in parents_2023.columns:
    parent_guess = next(
        (c for c in parents_2023.columns if re.search(r"parent", str(c), flags=re.IGNORECASE)),
        None,
    )
    if parent_guess:
        parents_2023 = parents_2023.rename(columns={parent_guess: "parent_name"})
        print(f"[info] inferred parent_name from column: {parent_guess}")

if "facility_id" not in parents_2023.columns:
    facility_guess = next(
        (c for c in parents_2023.columns if re.search(r"facility|ghgrp", str(c), flags=re.IGNORECASE)),
        None,
    )
    if facility_guess:
        parents_2023 = parents_2023.rename(columns={facility_guess: "facility_id"})
        print(f"[info] inferred facility_id from column: {facility_guess}")

print("Resolved parent columns:", [c for c in ["facility_id", "parent_name", "ownership_pct"] if c in parents_2023.columns])

if "facility_id" not in parents_2023.columns or "parent_name" not in parents_2023.columns:
    raise RuntimeError(
        "Parent workbook is missing required columns after normalization. "
        "Inspect parents_2023.columns and extend PARENT_ALIASES."
)

# Some rows have multiple parents (joint ownership) — keep all and weight by ownership_pct.
if "ownership_pct" in parents_2023.columns:
    parents_2023["ownership_share"] = (
        pd.to_numeric(parents_2023["ownership_pct"], errors="coerce") / 100.0
).fillna(1.0)
else:
    print("[warn] ownership percent column not found; defaulting ownership_share=1.0")
    parents_2023["ownership_share"] = 1.0

# Quick sanity check: ownership shares per facility should sum to ~1.0
share_check = parents_2023.groupby("facility_id")["ownership_share"].sum().describe()
print("Per-facility ownership-share sum (should be near 1.0):")
print(share_check)

[info] inferred parent_name from column: PARENT COMPANY NAME
Resolved parent columns: ['facility_id', 'parent_name', 'ownership_pct']
Per-facility ownership-share sum (should be near 1.0):
count    8106.000000
mean        0.999884
std         0.006806
min         0.500000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000400
Name: ownership_share, dtype: float64


In [10]:
# --- attribute 2023 emissions to parents (weighted) ---
power_with_parent = power_facilities.merge(
    parents_2023[["facility_id", "parent_name", "ownership_share"]],
    on="facility_id",
    how="left",
)
# Facilities with no parent record retain a single "self" attribution to facility_name as a placeholder.
power_with_parent["parent_name"] = power_with_parent["parent_name"].fillna(
    power_with_parent["facility_name"].astype(str) + " (NO_PARENT_RECORD)"
)
power_with_parent["ownership_share"] = power_with_parent["ownership_share"].fillna(1.0)
power_with_parent["attributed_co2e"] = (
    pd.to_numeric(power_with_parent["co2e_tons"], errors="coerce")
    * power_with_parent["ownership_share"]
)

# Roll up to parent level
parent_2023 = (
    power_with_parent.groupby("parent_name", as_index=False)
    .agg(
        facilities=("facility_id", "nunique"),
        states=("state", lambda s: s.dropna().nunique()),
        attributed_co2e=("attributed_co2e", "sum"),
    )
    .sort_values("attributed_co2e", ascending=False)
    .reset_index(drop=True)
)

# Lock top 50 — but first exclude orphaned NO_PARENT_RECORD facilities.
# These are retired plants whose parent record is missing from the 2023 parent
# workbook; they are not parent companies and should not be ranked as such.
COHORT_N = 50
orphan_mask = parent_2023["parent_name"].astype(str).str.contains("NO_PARENT_RECORD", na=False)
n_orphans_removed = orphan_mask.sum()
parent_2023_clean = parent_2023.loc[~orphan_mask].reset_index(drop=True)
print(f"Excluded {n_orphans_removed} NO_PARENT_RECORD entries from ranking pool.")

cohort_top50 = parent_2023_clean.head(COHORT_N).copy()
cohort_top50["rank"] = range(1, len(cohort_top50) + 1)
cohort_top50 = cohort_top50[["rank", "parent_name", "facilities", "states", "attributed_co2e"]]

cohort_top50_path = DATA_PROCESSED / "cohort_top50.csv"
cohort_top50.to_csv(cohort_top50_path, index=False)
print(f"\\nWrote: {cohort_top50_path}")
print(f"Top-50 share of 2023 power-sector emissions (denominator excludes orphans): "
      f"{cohort_top50['attributed_co2e'].sum() / parent_2023_clean['attributed_co2e'].sum():.1%}")
cohort_top50.head(20)

Excluded 299 NO_PARENT_RECORD entries from ranking pool.
\nWrote: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/processed/cohort_top50.csv
Top-50 share of 2023 power-sector emissions (denominator excludes orphans): 71.8%


,rank,parent_name,facilities,states,attributed_co2e
0,1,Vistra Corp,36,12,8.593720e+07
1,2,THE SOUTHERN CO,26,4,7.586176e+07
2,3,DUKE ENERGY CORP,36,6,7.206160e+07
3,4,BERKSHIRE HATHAWAY INC,34,11,4.938425e+07
4,5,AMERICAN ELECTRIC POWER CO INC,24,9,4.878462e+07
5,6,CPN MANAGEMENT LP,43,14,4.767363e+07
6,7,NEXTERA ENERGY INC,19,5,4.230858e+07
7,8,US GOVERNMENT,19,4,3.802682e+07
8,9,ENTERGY CORP,25,4,3.783322e+07
9,10,XCEL ENERGY INC,25,6,3.524903e+07


In [11]:
# Also persist the underlying facility→parent attribution table for downstream joins.
attribution_path = DATA_PROCESSED / "power_facility_parent_attribution_2023.csv"
keep_cols = [
    "facility_id", "facility_name", "state", "city", "county", "lat", "lon",
    "subparts", "co2e_tons", "parent_name", "ownership_share", "attributed_co2e",
]
keep_cols = [c for c in keep_cols if c in power_with_parent.columns]
power_with_parent[keep_cols].to_csv(attribution_path, index=False)
print(f"Wrote: {attribution_path}  ({len(power_with_parent):,} rows)")

Wrote: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/processed/power_facility_parent_attribution_2023.csv  (1,614 rows)


## 3. EPA AMPD / CAMD CEMS — facility-unit-annual emissions, 2011–2024

CEMS (Continuous Emissions Monitoring System) data is collected hourly at the unit level for every fossil-fueled power plant subject to Acid Rain / Cross-State Air Pollution rules. It is the closest thing to direct, certified ground truth for power-sector emissions in the US — and the independent third leg that lets us validate GHGRP self-reports for the power sector. Notebook 03 will use it as the ground-truth anchor in the Linear Mixed-Effects diagnostic that quantifies Climate TRACE's structural bias.

### Endpoint choice — `streaming-services`, not `bulk-files`

EPA exposes CAMD data through two distinct APIs:

- **`camd-services/bulk-files`** serves raw ECMPS submission ZIPs (monitoring plans, QA tests, EDR submissions). The manifest has a 15-minute timeout and returns a degraded 1-entry sample on auth failure. This is the wrong endpoint for our use case.
- **`streaming-services/emissions/apportioned/annual`** serves clean facility-unit-annual emissions CSV with no record limit and is designed for programmatic access. This is what we use.

### Pull strategy

- **Endpoint:** `https://api.epa.gov/easey/streaming-services/emissions/apportioned/annual`
- **Auth:** `x-api-key` header (NOT a query parameter — the query-param form silently returns a degraded sample).
- **Parameter:** `year` as an array — EPA's API expects repeated query params (`?year=2011&year=2012&…`) rather than a date range or a single value.
- **Loop strategy:** one year per request. A multi-year batch returned HTTP 500 ("An internal issue occured" — undocumented per-call size limit on EPA's backend).
- **Throttling:** 2-second sleep between successful calls and exponential backoff (30s → 60s → 120s → 240s) on HTTP 429. EPA's effective short-window rate limit is tighter than the documented 1,000/hr.
- **Idempotence:** each year saves to `data/raw/cems/emissions_annual_{year}.csv`; re-running the cell skips any year already on disk. If any single year fails, just re-run the cell — only the missing years will retry.

### Output

After the loop, the validation cell stitches all 14 per-year files into `data/raw/cems/emissions_annual_2011_2024.csv` (~62,000 unit-year rows, 26 columns including `State`, `Facility ID`, `Unit ID`, `Year`, `CO2 Mass (short tons)`, `Gross Load (MWh)`, `Heat Input (mmBtu)`, `Primary Fuel Type`, `Unit Type`).

**Unit-conversion note for downstream notebooks:** CEMS reports CO₂ in *short tons*, while GHGRP and Climate TRACE both use *metric tons*. Notebook 02 converts CEMS to metric tons via the standard factor (1 short ton = 0.907185 metric tons) before any cross-source comparison.

### Registering for an API key

Free, instant: https://www.epa.gov/power-sector/cam-api-portal#request-api-key
Set the resulting key in `.env`:
```
EPA_CAMD_API_KEY=your-key
```

In [12]:
EPA_CAMD_API_KEY = os.environ.get("EPA_CAMD_API_KEY", "").strip()
CEMS_DIR = DATA_RAW / "cems"
CEMS_DIR.mkdir(parents=True, exist_ok=True)

CEMS_BEGIN_YEAR = 2011
CEMS_END_YEAR   = 2024
CEMS_OUT_CSV    = CEMS_DIR / f"emissions_annual_{CEMS_BEGIN_YEAR}_{CEMS_END_YEAR}.csv"

if not EPA_CAMD_API_KEY:
    print("[ACTION REQUIRED] EPA_CAMD_API_KEY env var is not set.")
    print("  1. Register at https://www.epa.gov/power-sector/cam-api-portal#request-api-key")
    print("  2. Add to .env:  EPA_CAMD_API_KEY=your-key")
    print("  3. Re-run cell 02 (env), then this cell.")
else:
    print(f"EPA_CAMD_API_KEY present ({EPA_CAMD_API_KEY[:6]}…) — will hit streaming-services/emissions/apportioned/annual.")
    print(f"Years: {CEMS_BEGIN_YEAR}–{CEMS_END_YEAR}")
    print(f"Output: {CEMS_OUT_CSV}")

EPA_CAMD_API_KEY present (i71nqg…) — will hit streaming-services/emissions/apportioned/annual.
Years: 2011–2024
Output: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/raw/cems/emissions_annual_2011_2024.csv


In [24]:
# Probe: fetch a single year (2023) to confirm auth and schema before pulling the full panel.
# This is a CHEAP request that returns CSV; if it succeeds we know the credentials and endpoint work.
PROBE_URL = "https://api.epa.gov/easey/streaming-services/emissions/apportioned/annual"

if EPA_CAMD_API_KEY:
    probe_params = {"year": 2023}
    probe_headers = {"x-api-key": EPA_CAMD_API_KEY, "Accept": "text/csv"}
    print(f"GET {PROBE_URL}?year=2023")
    try:
        r = requests.get(PROBE_URL, params=probe_params, headers=probe_headers, timeout=120, stream=False)
        print(f"  HTTP {r.status_code}  ({len(r.content):,} bytes  content-type={r.headers.get('content-type','?')})")
        if r.status_code == 200:
            # Streaming-services may return CSV or JSON depending on Accept header / endpoint version.
            text = r.text
            first_lines = text.splitlines()[:5]
            print("\nFirst 5 lines of response:")
            for line in first_lines:
                print(" ", line[:200])
            if "," in (first_lines[0] if first_lines else ""):
                cols = first_lines[0].split(",")
                print(f"\nDetected CSV header: {len(cols)} columns")
                print("  Sample columns:", cols[:8])
            cems_probe_ok = True
        else:
            print(f"\n[ERROR] non-200 response. Body (first 800 chars):")
            print(r.text[:800])
            cems_probe_ok = False
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        cems_probe_ok = False
else:
    print("(skipped — no API key)")
    cems_probe_ok = False

GET https://api.epa.gov/easey/streaming-services/emissions/apportioned/annual?year=2023
  HTTP 200  (960,028 bytes  content-type=text/csv)

First 5 lines of response:
  "State","Facility Name","Facility ID","Unit ID","unit_id","Associated Stacks","Year","Operating Time Count","Sum of the Operating Time","Gross Load (MWh)","Steam Load (1000 lb)","SO2 Mass (short tons)
  "AL","Barry",3,"1",1,"CS0AAN",2023,575,571,16049.5,,0.068,0.001,13509.825,0.0592,7.514,0.0567,227316.475,"Pipeline Natural Gas",,"Tangentially-fired",,"Low NOx Burner Technology w/ Closed-coupled OFA|
  "AL","Barry",3,"2",2,"CS0AAN",2023,562,555.75,15528,,0.064,0.001,12757.45,0.0592,7.454,0.057,214656.475,"Pipeline Natural Gas",,"Tangentially-fired",,"Low NOx Burner Technology w/ Closed-coupled OFA|S
  "AL","Barry",3,"4",4,,2023,6189,6182.75,601338,,1.974,0.001,391039.55,0.059,386.123,0.118,6580043.725,"Pipeline Natural Gas",,"Tangentially-fired",,"Low NOx Burner Technology w/ Separated OFA|Selectiv
  "AL","Barry",3,"5",

In [26]:
# Full pull: 2011-2024 facility-unit-annual emissions, ONE year per request.
# EPA enforces a short-window rate limit that's tighter than the documented 1,000/hr.
# In a clean run we hit HTTP 429 ("OVER_RATE_LIMIT") on roughly every 3rd-4th call.
# Fix: (a) longer inter-call sleep, (b) automatic retry on 429 honoring Retry-After.
# The per-year file is idempotent, so re-running this cell only fetches missing years.

CEMS_FULL_URL    = "https://api.epa.gov/easey/streaming-services/emissions/apportioned/annual"
CEMS_INTER_SLEEP = 2.0          # seconds between successful calls
CEMS_MAX_RETRIES = 4            # retries when we hit HTTP 429
CEMS_BACKOFFS    = [30, 60, 120, 240]   # seconds to wait if Retry-After is absent


def cems_fetch_year(year: int) -> tuple[bool, str]:
    """Fetch one year of apportioned-annual CSV; retries on 429 with backoff."""
    out_path = CEMS_DIR / f"emissions_annual_{year}.csv"
    if out_path.exists() and out_path.stat().st_size > 0:
        return True, f"[skip] {out_path.name} ({out_path.stat().st_size:,} B)"

    params  = {"year": year}
    headers = {"x-api-key": EPA_CAMD_API_KEY, "Accept": "text/csv"}

    for attempt in range(CEMS_MAX_RETRIES + 1):
        try:
            with requests.get(CEMS_FULL_URL, params=params, headers=headers,
                              stream=True, timeout=300) as r:
                if r.status_code == 200:
                    bytes_written = 0
                    with open(out_path, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1 << 16):
                            if chunk:
                                f.write(chunk)
                                bytes_written += len(chunk)
                    return True, f"[ok   {year}] {out_path.name} ({bytes_written:,} B)"

                if r.status_code == 429:
                    # honor Retry-After if present, else use exponential backoff
                    ra = r.headers.get("Retry-After")
                    try:
                        wait = int(ra) if ra else CEMS_BACKOFFS[min(attempt, len(CEMS_BACKOFFS)-1)]
                    except ValueError:
                        wait = CEMS_BACKOFFS[min(attempt, len(CEMS_BACKOFFS)-1)]
                    if attempt < CEMS_MAX_RETRIES:
                        print(f"  [429 {year}] rate-limited; sleeping {wait}s (attempt {attempt+1}/{CEMS_MAX_RETRIES})…")
                        time.sleep(wait)
                        continue
                    return False, f"[err {year}] HTTP 429 after {CEMS_MAX_RETRIES} retries"

                # other non-200
                return False, f"[err {year}] HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            return False, f"[err {year}] {type(e).__name__}: {e}"
    return False, f"[err {year}] exhausted retries"


cems_year_results = []
if EPA_CAMD_API_KEY and cems_probe_ok:
    for year in tqdm(range(CEMS_BEGIN_YEAR, CEMS_END_YEAR + 1), desc="CEMS years"):
        ok, msg = cems_fetch_year(year)
        cems_year_results.append({"year": year, "ok": ok, "msg": msg})
        print(msg)
        time.sleep(CEMS_INTER_SLEEP)   # polite inter-call throttle

    n_ok = sum(1 for r in cems_year_results if r["ok"])
    print(f"\nFinished: {n_ok}/{len(cems_year_results)} years on disk.")
    missing = [r["year"] for r in cems_year_results if not r["ok"]]
    if missing:
        print(f"Still missing: {missing} — re-run this cell to retry just these.")
elif EPA_CAMD_API_KEY:
    print("(skipped — probe failed; resolve cell 17 first)")
else:
    print("(skipped — no API key)")

CEMS years:   0%|          | 0/14 [00:00<?, ?it/s]

[skip] emissions_annual_2011.csv (1,125,073 B)


CEMS years:   7%|▋         | 1/14 [00:02<00:26,  2.01s/it]

[skip] emissions_annual_2012.csv (1,130,320 B)


CEMS years:  14%|█▍        | 2/14 [00:04<00:24,  2.01s/it]

...

[ok   2023] emissions_annual_2023.csv (960,028 B)


CEMS years:  93%|█████████▎| 13/14 [00:32<00:02,  2.68s/it]

[skip] emissions_annual_2024.csv (944,784 B)


CEMS years: 100%|██████████| 14/14 [00:34<00:00,  2.49s/it]


Finished: 14/14 years on disk.


In [27]:
# Validation: read every per-year CSV in data/raw/cems/, concatenate to a single
# DataFrame, and report on the combined panel. Also write a stitched CSV for
# downstream notebooks that prefer a single input.
per_year_files = sorted(CEMS_DIR.glob("emissions_annual_*.csv"))
print(f"Per-year CSVs found: {len(per_year_files)}")

if not per_year_files:
    print("(no per-year CSVs on disk — review cell 18 output)")
else:
    cems_frames = []
    for p in per_year_files:
        try:
            df = pd.read_csv(p, low_memory=False)
            cems_frames.append(df)
            print(f"  {p.name}: {len(df):,} rows × {df.shape[1]} cols")
        except Exception as e:
            print(f"  {p.name}: [warn] {e}")

    if cems_frames:
        cems_all = pd.concat(cems_frames, ignore_index=True)
        # write stitched panel for downstream use
        stitched = CEMS_DIR / "emissions_annual_2011_2024.csv"
        cems_all.to_csv(stitched, index=False)
        print(f"\nStitched panel: {stitched.name}  ({len(cems_all):,} rows)")

        print(f"\nColumns ({len(cems_all.columns)}):")
        for c in cems_all.columns:
            print(f"  - {c}")

        year_col = next((c for c in cems_all.columns if c.lower() in ("year", "reportingyear")), None)
        if year_col:
            print(f"\nRows per year ({year_col}):")
            print(cems_all[year_col].value_counts().sort_index().to_string())

        fac_col = next((c for c in cems_all.columns if c.lower() in ("facility id", "facilityid")), None)
        if fac_col:
            print(f"\nUnique facilities across panel: {cems_all[fac_col].nunique():,}")

Per-year CSVs found: 14
  emissions_annual_2011.csv: 4,825 rows × 26 cols
  emissions_annual_2012.csv: 4,843 rows × 26 cols
  emissions_annual_2013.csv: 4,770 rows × 26 cols
  emissions_annual_2014.csv: 4,703 rows × 26 cols
  emissions_annual_2015.csv: 4,686 rows × 26 cols
  emissions_annual_2016.csv: 4,545 rows × 26 cols
  emissions_annual_2017.csv: 4,383 rows × 26 cols
  emissions_annual_2018.csv: 4,389 rows × 26 cols
  emissions_annual_2019.csv: 4,319 rows × 26 cols
  emissions_annual_2020.csv: 4,254 rows × 26 cols
  emissions_annual_2021.csv: 4,178 rows × 26 cols
  emissions_annual_2022.csv: 4,131 rows × 26 cols
  emissions_annual_2023.csv: 4,090 rows × 26 cols
  emissions_annual_2024.csv: 4,024 rows × 26 cols

Stitched panel: emissions_annual_2011_2024.csv  (62,140 rows)

Columns (26):
  - State
  - Facility Name
  - Facility ID
  - Unit ID
  - unit_id
  - Associated Stacks
  - Year
  - Operating Time Count
  - Sum of the Operating Time
  - Gross Load (MWh)
  - Steam Load (1000 lb

## 4. Climate TRACE — USA country package (one-time manual download)

Climate TRACE (https://climatetrace.org) publishes asset-level emissions estimates derived from satellite imagery, activity data, and emission-factor models. For our framework, it is the **Verify** axis — an independent third estimate that we compare against GHGRP self-reports and CEMS direct measurement.

### Download approach

The current Climate TRACE versioned URLs (v04, v05) on `downloads.climatetrace.org` are version-fragile and our auto-probe of likely paths all returned 404. We therefore use a **one-time manual download** with a fixed save-path convention:

1. Visit https://climatetrace.org/data and select the current USA country package (CSV bundle, all 10 sectors).
2. Save the downloaded zip as exactly `data/raw/climate_trace/climate_trace_us_power.zip`.
3. Re-run cells 21 and 22. Cell 21 prints `[skip]` because the file is already on disk; cell 22 unpacks it.
4. The extracted folder is `data/raw/climate_trace/climate_trace_us_power/DATA/` with one subdirectory per sector. **We use `DATA/power/`** for our analysis.

You can also set `CLIMATE_TRACE_URL=https://…` in `.env` if a stable direct download URL becomes available; the cell will use it.

### Honesty note on what Climate TRACE actually is

Climate TRACE's estimates for U.S. power plants lean heavily on activity data and EIA cross-references, not raw satellite imagery alone. We treat Climate TRACE as an **independent, model-based, remote-sensing-informed estimate** — not pristine remote sensing. This caveat appears verbatim in the working paper's methodology section, and is why the SCVS (Satellite Cross-Validation Score) is weighted at only 0.2 in the composite CCCS — Climate TRACE is a verification check, not a primary measurement.

In [16]:
CT_DIR = DATA_RAW / "climate_trace"

# Try the current sector-package URL pattern first. If 404, fall back to country package.
# Climate TRACE has moved its download URLs over time. We try several known
# patterns; if all fail, we print explicit manual-download instructions.
# Set CT_OVERRIDE_URL via env var to bypass probing entirely.
CT_OVERRIDE_URL = os.environ.get("CLIMATE_TRACE_URL", "").strip()
CT_CANDIDATES = [CT_OVERRIDE_URL] if CT_OVERRIDE_URL else [
    # Country package — non-forest CSV bundle, current 2024/2025 release pattern
    "https://downloads.climatetrace.org/v04/country-packages/non-forest/USA.zip",
    "https://downloads.climatetrace.org/v03/country-packages/non-forest/USA.zip",
    # Sector subset — power-only
    "https://downloads.climatetrace.org/v04/sector-packages/power.zip",
    "https://downloads.climatetrace.org/v04/sector-packages/electricity-generation.zip",
    # Legacy underscore variants (probed last)
    "https://downloads.climatetrace.org/v05/sector_packages/electricity-generation.zip",
    "https://downloads.climatetrace.org/v04/sector_packages/electricity-generation.zip",
]
CT_CANDIDATES = [u for u in CT_CANDIDATES if u]   # drop empties

ct_zip = CT_DIR / "climate_trace_us_power.zip"
ct_used_url = None

if ct_zip.exists():
    print(f"[present] {ct_zip.name} ({ct_zip.stat().st_size:,} B) — skipping download")
else:
    for url in CT_CANDIDATES:
        try:
            print(f"  trying: {url}")
            http_download(url, ct_zip)
            ct_used_url = url
            break
        except requests.HTTPError as e:
            print(f"  [miss] {e.response.status_code} on {url}")
        except Exception as e:
            print(f"  [error] {e}")
    if not ct_zip.exists():
        print("[ACTION REQUIRED] could not auto-download Climate TRACE.")
        print(f"  Visit https://climatetrace.org/data, pick the latest US power-sector package,")
        print(f"  and save it as {ct_zip}")

print(f"Used URL: {ct_used_url}")

[present] climate_trace_us_power.zip (351,569,742 B) — skipping download
Used URL: None


In [17]:
# Unpack the zip (no-op if already unpacked).
if ct_zip.exists():
    unpack_dir = CT_DIR / ct_zip.stem
    if unpack_dir.exists() and any(unpack_dir.iterdir()):
        print(f"[present] {unpack_dir} already populated")
    else:
        unpack_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ct_zip) as zf:
            zf.extractall(unpack_dir)
        print(f"Extracted to {unpack_dir}")
    # list top-level contents
    for p in sorted(unpack_dir.iterdir())[:20]:
        print(" •", p.name)

[present] /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/raw/climate_trace/climate_trace_us_power already populated
 • ABOUT_THE_DATA
 • DATA


## 5. SBTi — public corporate targets dashboard

The Science Based Targets initiative (https://sciencebasedtargets.org) publishes a public Excel export of every validated and committed corporate climate target. This is the canonical input for **four of the five PQS pillars**:

- **Ambition** — percent reduction commitment + pathway alignment (1.5°C / well-below-2°C / 2°C)
- **Specificity** — presence of defined base year and target year
- **Time Horizon Structure** — near-term + long-term targets, interim milestones
- **Verification & Governance** — SBTi validation status (validated targets carry external scientific review)

The fifth pillar (**Mechanism Clarity**) is sourced from SEC 10-K filings via regex/keyword density scoring in Notebook 04 — that pillar is what the NLP layer earns.

### Files downloaded

- `companies-excel.xlsx` (~2.1 MB) — one row per company with metadata
- `targets-excel.xlsx` (~5.4 MB) — one row per target (companies can have multiple targets for different scopes / horizons)

### Fallbacks

If the auto-download URL ever returns a non-XLSX response (e.g., HTML if SBTi changes their export endpoint), the cell prints `[ACTION REQUIRED]` with instructions to manually download via the dashboard's Export button at https://sciencebasedtargets.org/target-dashboard.

In [18]:
SBTI_DIR = DATA_RAW / "sbti"

SBTI_URLS = {
    # Primary (May 2026 verified endpoint names)
    "companies-excel.xlsx": [
        "https://sciencebasedtargets.org/download/excel",
    ],
    "targets-excel.xlsx": [
        "https://sciencebasedtargets.org/download/targets-excel",
        "https://sciencebasedtargets.org/download/target-excel",
    ],
}

sbti_status = []
for fname, urls in SBTI_URLS.items():
    p = SBTI_DIR / fname
    if p.exists():
        print(f"[present] {fname} ({p.stat().st_size:,} B)")
        sbti_status.append({"file": fname, "status": "present", "url": urls[0]})
        continue

    downloaded = False
    last_error = None
    for url in urls:
        try:
            http_download(url, p, headers={"User-Agent": "Mozilla/5.0 (research)"})
            sbti_status.append({"file": fname, "status": "downloaded", "url": url})
            downloaded = True
            break
        except Exception as e:
            last_error = e
            print(f"  [miss] {url} -> {e}")

    if not downloaded:
        print(f"  [ACTION REQUIRED] SBTi auto-download failed: {last_error}")
        print(f"  Visit https://sciencebasedtargets.org/target-dashboard -> Export, save as {p}")
        sbti_status.append({"file": fname, "status": "MISSING", "url": urls[0]})

pd.DataFrame(sbti_status)

[present] companies-excel.xlsx (2,156,432 B)
[present] targets-excel.xlsx (5,443,217 B)


,file,status,url
0,companies-excel.xlsx,present,https://sciencebasedtargets.org/download/excel
1,targets-excel.xlsx,present,https://sciencebasedtargets.org/download/targe...


## 6. SEC EDGAR — most recent 10-K for each cohort parent that is a registrant

We pull the most recent annual 10-K for each cohort parent that files with the SEC. The 10-K is the source for:
- **Item 1A (Risk Factors)** — climate-risk language and disclosure quality
- **Item 7 (MD&A)** — capex commitments and operational-decarbonization narrative
- **Revenue** — for the EPA-grounded WACI denominator in a later notebook

**No API key needed.** SEC requires a polite `User-Agent` header identifying the requester by name + email per their fair-use policy. We read it from `SEC_EDGAR_UA` in `.env`.

### Matching parents to CIKs

The matching step (parent name → SEC Central Index Key) is fuzzy and inevitably imperfect. We apply a three-layer guard against false positives:

1. **`DO_NOT_MATCH` blocklist** — 25 known non-registrants are explicitly returned as unmatched. This includes:
   - Cooperatives (Basin Electric, Tri-State G&T, East Kentucky Power Co-op, etc.)
   - Public-power authorities (Salt River Project, Santee Cooper, OPPD, NPPD, CPS Energy, LCRA, etc.)
   - PE-owned holdcos and joint ventures (ArcLight, Lightstone Generation, LS Power Development, CPN Management, Puget Holdings, Cleco Corporate Holdings)
   - Defunct registrants (GenOn Energy — went private 2018, has no current 10-K)
   - Subsidiaries of foreign parents (UNS Energy → Fortis; no separate 10-K)
   - Federal entities (`US GOVERNMENT` = TVA + federal facilities; TVA has bonds but no 10-K)

2. **Manual positive overrides** — EPA's spelling differs from SEC's title (e.g., `THE SOUTHERN CO` vs `Southern Co`). 23 explicit overrides cover known utilities so they bypass fuzzy matching entirely.

3. **Fuzzy matcher (last resort)** — uses `difflib.get_close_matches` with cutoff 0.92 (tightened from the 0.85 default) AND requires the first token of the parent name (after punctuation normalization) to match the first token of the SEC title. This blocks the kinds of false positives we saw in early runs (`GENON ENERGY INC` → `MGE ENERGY INC`; `UNS ENERGY CORP` → `US ENERGY CORP`).

### Expected coverage

Of 50 cohort parents, approximately 22–23 are SEC registrants and have their 10-Ks downloaded; the remaining 27–28 are correctly identified as non-registrants and blocked. This is a real-world fact about the U.S. power sector — many large emitters are cooperatives or public-power authorities outside SEC's filing scope — and is documented as a limitation of any PQS Pillar-5 (Mechanism Clarity) coverage analysis in the working paper.

### Outputs

- `data/processed/parent_to_cik.csv` — every cohort parent with CIK (or `None`) and `match_method` (`override` / `exact` / `fuzzy:…` / `blocked:do_not_match` / `unmatched`)
- `data/processed/tenk_filings_index.csv` — accession numbers, primary document, filing date, document URL for each downloaded 10-K
- `data/raw/sec_10k/{parent_name}/` — the actual 10-K HTML primary documents

In [19]:
SEC_UA = os.environ.get("SEC_EDGAR_UA", "").strip() or "Souvik Mandal souvikces@gmail.com"
SEC_HEADERS = {"User-Agent": SEC_UA, "Accept-Encoding": "gzip, deflate"}
TENK_DIR = DATA_RAW / "sec_10k"

# Download ticker → CIK lookup (small file, ~12k tickers)
tickers_path = DATA_EXTERNAL / "sec_company_tickers.json"
if not tickers_path.exists():
    http_download("https://www.sec.gov/files/company_tickers.json",
                  tickers_path, headers=SEC_HEADERS)

tickers_raw = json.loads(tickers_path.read_text())
# JSON is keyed by integer-as-string; values are {cik_str, ticker, title}
ticker_df = pd.DataFrame(tickers_raw.values())
ticker_df["cik_padded"] = ticker_df["cik_str"].astype(int).astype(str).str.zfill(10)
print(f"SEC ticker registry: {len(ticker_df):,} companies")
ticker_df.head(3)

SEC ticker registry: 10,371 companies


,cik_str,ticker,title,cik_padded
0,1045810,NVDA,NVIDIA CORP,0001045810
1,1652044,GOOGL,Alphabet Inc.,0001652044
2,320193,AAPL,Apple Inc.,0000320193


In [20]:
# Manual overrides for parents whose EPA names won't fuzzy-match cleanly.
# Extend this dict as you encounter ambiguous cases.
# NEGATIVE overrides: parents that should NEVER be matched (no public 10-K exists)
# This blocks fuzzy false-positives such as GENON ENERGY INC -> MGE ENERGY INC.
DO_NOT_MATCH = {
    "GENON ENERGY INC",                       # went private 2018, no SEC registrant
    "UNS ENERGY CORP",                        # subsidiary of Fortis, no separate 10-K
    "ARCLIGHT ENERGY PARTNERS FUND VII LP",   # PE fund, not registrant
    "ARCLIGHT CAPITAL HOLDINGS LLC",          # PE fund, not registrant
    "LIGHTSTONE GENERATION LLC",              # PE JV, not registrant
    "LS Power Development, LLC",              # private holdco
    "CPN MANAGEMENT LP",                      # Calpine subsidiary, parent is private
    "PRAIRIE STATE ENERGY CAMPUS MANAGEMENT CO",  # cooperative venture
    "PUGET HOLDINGS LLC",                     # private holdco, no 10-K
    "CLECO CORPORATE HOLDINGS LLC",           # subsidiary of Macquarie infra, no 10-K
    "REMC ASSETS LP",                         # private LP
    # public power / cooperatives — no SEC 10-K obligation
    "BASIN ELECTRIC POWER COOPERATIVE",
    "BUCKEYE POWER INC",
    "ASSOCIATED ELECTRIC COOPERATIVE INC",
    "OGLETHORPE POWER CORP",
    "TRI-STATE GENERATION & TRANSMISSION ASSOC INC",
    "EAST KENTUCKY POWER COOPERATIVE INC",
    "SEMINOLE ELECTRIC COOPERATIVE INC",
    "ARKANSAS ELECTRIC COOPERATIVE CORP",
    "OMAHA PUBLIC POWER DISTRICT",
    "NEBRASKA PUBLIC POWER DISTRICT",
    "CPS ENERGY",
    "SOUTH CAROLINA PUBLIC SERVICE AUTHORITY",
    "SALT RIVER PROJECT AGRICULTURAL IMPROVEMENT & POWER DISTRICT",
    "LOWER COLORADO RIVER AUTHORITY",
    "US GOVERNMENT",                          # TVA + federal — TVA has bonds but no 10-K
}

# POSITIVE overrides: explicit EPA-name -> CIK mappings, using EPA-style spellings.
MANUAL_CIK_OVERRIDES = {
    # EPA-style upper-case names (these will match exactly to cohort entries)
    "THE SOUTHERN CO":                        "0000092122",
    "ENTERGY CORP":                           "0000065984",
    "TECO ENERGY INC":                        "0001022909",
    # legacy title-case keys (kept for backward compatibility with prior runs)
    "NextEra Energy Inc":                "0000753308",
    "Duke Energy Corporation":           "0001326160",
    "Southern Company":                  "0000092122",
    "American Electric Power Company":   "0000004904",
    "Vistra Corp":                       "0001692819",
    "Exelon Corporation":                "0001109357",
    "Dominion Energy Inc":               "0000715957",
    "Berkshire Hathaway Energy Company": "0001081316",  # parent: Berkshire Hathaway
    "PPL Corporation":                   "0000922224",
    "FirstEnergy Corp":                  "0001031296",
    "Constellation Energy Corporation":  "0001868275",
    "DTE Energy Company":                "0000936340",
    "Xcel Energy Inc":                   "0000072903",
    "Edison International":              "0000827052",
    "Sempra Energy":                     "0001032208",
    "Entergy Corporation":               "0000065984",
    "NRG Energy Inc":                    "0001013871",
    "PG&E Corporation":                  "0001004980",
    "Eversource Energy":                 "0000072741",
    "Consolidated Edison Inc":           "0001047862",
}

from difflib import get_close_matches


def _norm_for_match(s: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace. Used on BOTH sides
    of the comparison so 'Evergy, Inc.' and 'EVERGY INC' both reduce to 'evergy inc'."""
    s = re.sub(r"[^a-zA-Z0-9 ]", " ", str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()


def fuzzy_cik(parent_name: str, ticker_df: pd.DataFrame, override: dict) -> tuple[Optional[str], str]:
    # 0. block known non-registrants (PE holdcos, public power, defunct registrants)
    if parent_name in DO_NOT_MATCH:
        return None, "blocked:do_not_match"
    # 1. exact override
    if parent_name in override:
        return override[parent_name], "override"
    # 2. normalized exact match (handles 'Inc.' vs 'INC')
    norm = _norm_for_match(parent_name)
    titles_norm = ticker_df["title"].astype(str).map(_norm_for_match)
    exact = ticker_df.loc[titles_norm == norm]
    if len(exact):
        return exact.iloc[0]["cik_padded"], "exact"
    # 3. close match — first-token gate operates on NORMALIZED titles
    #    (kills UNS->US-ENERGY because first tokens differ; allows EVERGY->Evergy, Inc.)
    first_token = norm.split()[0] if norm.split() else ""
    mask = titles_norm.str.startswith(first_token + " ") | (titles_norm == first_token)
    candidate_titles_norm = titles_norm[mask].tolist()
    matches = get_close_matches(norm, candidate_titles_norm, n=1, cutoff=0.92)
    if matches:
        hit = ticker_df.loc[titles_norm == matches[0]].iloc[0]
        return hit["cik_padded"], f"fuzzy:{matches[0]}"
    return None, "unmatched"


# load cohort
cohort_top50 = pd.read_csv(DATA_PROCESSED / "cohort_top50.csv")

parent_cik = []
for name in cohort_top50["parent_name"]:
    cik, method = fuzzy_cik(name, ticker_df, MANUAL_CIK_OVERRIDES)
    parent_cik.append({"parent_name": name, "cik": cik, "match_method": method})

parent_cik_df = pd.DataFrame(parent_cik)
parent_cik_df.to_csv(DATA_PROCESSED / "parent_to_cik.csv", index=False)
unmatched = parent_cik_df[parent_cik_df["cik"].isna()]
print(f"Matched:   {parent_cik_df['cik'].notna().sum()} / {len(parent_cik_df)}")
print(f"Unmatched: {len(unmatched)} — review these:")
print(unmatched.to_string(index=False))

Matched:   26 / 50
Unmatched: 24 — review these:
                                                 parent_name  cik         match_method
                                           CPN MANAGEMENT LP None blocked:do_not_match
                                               US GOVERNMENT None blocked:do_not_match
                                   LIGHTSTONE GENERATION LLC None blocked:do_not_match
                            BASIN ELECTRIC POWER COOPERATIVE None blocked:do_not_match
SALT RIVER PROJECT AGRICULTURAL IMPROVEMENT & POWER DISTRICT None blocked:do_not_match
                         ASSOCIATED ELECTRIC COOPERATIVE INC None blocked:do_not_match
                     SOUTH CAROLINA PUBLIC SERVICE AUTHORITY None blocked:do_not_match
                   PRAIRIE STATE ENERGY CAMPUS MANAGEMENT CO None blocked:do_not_match
                                   LS Power Development, LLC None blocked:do_not_match
                                                  CPS ENERGY None blocked:do_not_

In [21]:
# Pull the most recent 10-K for each matched parent.
def latest_10k_for_cik(cik_padded: str, headers: dict) -> Optional[dict]:
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    sub = r.json()
    forms = sub.get("filings", {}).get("recent", {})
    df = pd.DataFrame(forms)
    if df.empty:
        return None
    df10 = df[df["form"] == "10-K"]
    if df10.empty:
        return None
    row = df10.iloc[0]   # most recent
    accession_no_dashes = row["accessionNumber"].replace("-", "")
    return {
        "cik": cik_padded,
        "accession": row["accessionNumber"],
        "primary_doc": row["primaryDocument"],
        "filing_date": row["filingDate"],
        "report_date": row["reportDate"],
        "doc_url": f"https://www.sec.gov/Archives/edgar/data/{int(cik_padded)}/"
                   f"{accession_no_dashes}/{row['primaryDocument']}",
    }

filings = []
for row in tqdm(parent_cik_df.dropna(subset=["cik"]).itertuples(index=False),
                total=parent_cik_df["cik"].notna().sum(), desc="10-K lookup"):
    try:
        info = latest_10k_for_cik(row.cik, SEC_HEADERS)
        if info:
            info["parent_name"] = row.parent_name
            filings.append(info)
            time.sleep(0.12)   # be polite to SEC
    except Exception as e:
        print(f"  [warn] {row.parent_name}: {e}")

filings_df = pd.DataFrame(filings)
print(f"Resolved 10-K filings for {len(filings_df)} parents")
filings_df.head(5)

10-K lookup: 100%|██████████| 26/26 [00:11<00:00,  2.20it/s]

Resolved 10-K filings for 25 parents


,cik,accession,primary_doc,filing_date,report_date,doc_url,parent_name
0,0001692819,0001692819-26-000006,vistra-20251231.htm,2026-02-27,2025-12-31,https://www.sec.gov/Archives/edgar/data/169281...,Vistra Corp
1,0000092122,0000092122-26-000006,so-20251231.htm,2026-02-19,2025-12-31,https://www.sec.gov/Archives/edgar/data/92122/...,THE SOUTHERN CO
2,0001326160,0001326160-26-000014,duk-20251231.htm,2026-02-26,2025-12-31,https://www.sec.gov/Archives/edgar/data/132616...,DUKE ENERGY CORP
3,0001067983,0001193125-26-083899,brka-20251231.htm,2026-03-02,2025-12-31,https://www.sec.gov/Archives/edgar/data/106798...,BERKSHIRE HATHAWAY INC
4,0000004904,0000004904-26-000013,aep-20251231.htm,2026-02-12,2025-12-31,https://www.sec.gov/Archives/edgar/data/4904/0...,AMERICAN ELECTRIC POWER CO INC


In [22]:
# Download the actual 10-K primary document (HTML) for each parent.
for f in tqdm(filings, desc="10-K download"):
    out_dir = TENK_DIR / f["parent_name"].replace("/", "_")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{f['filing_date']}__{f['primary_doc']}"
    try:
        http_download(f["doc_url"], out_path, headers=SEC_HEADERS)
        time.sleep(0.12)
    except Exception as e:
        print(f"  [warn] {f['parent_name']}: {e}")

filings_df.to_csv(DATA_PROCESSED / "tenk_filings_index.csv", index=False)
print(f"\\nWrote: {DATA_PROCESSED / 'tenk_filings_index.csv'}")

10-K download:   4%|▍         | 1/25 [00:00<00:03,  8.00it/s]

  [skip] 2026-02-27__vistra-20251231.htm already on disk (5,587,052 bytes)
  [skip] 2026-02-19__so-20251231.htm already on disk (15,639,198 bytes)


10-K download:  12%|█▏        | 3/25 [00:00<00:02,  7.91it/s]

  [skip] 2026-02-26__duk-20251231.htm already on disk (16,080,176 bytes)
  [skip] 2026-03-02__brka-20251231.htm already on disk (10,396,820 bytes)


10-K download:  20%|██        | 5/25 [00:00<00:02,  7.87it/s]

...

  [skip] 2026-02-26__tln-20251231.htm already on disk (3,992,782 bytes)
  [skip] 2026-02-24__ceg-20251231.htm already on disk (4,811,218 bytes)


10-K download:  92%|█████████▏| 23/25 [00:02<00:00,  7.98it/s]

  [skip] 2026-02-18__oge-20251231.htm already on disk (9,001,059 bytes)
  [skip] 2026-02-26__mpc-20251231.htm already on disk (4,001,121 bytes)


10-K download: 100%|██████████| 25/25 [00:03<00:00,  7.95it/s]

  [skip] 2026-02-18__xom-20251231.htm already on disk (5,591,068 bytes)
\nWrote: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/processed/tenk_filings_index.csv


## 7. Net Zero Tracker — public corporate climate-commitment registry

Net Zero Tracker (https://zerotracker.net) is a free public database maintained by a consortium of Energy & Climate Intelligence Unit, Data-Driven EnviroLab, NewClimate Institute, and Oxford Net Zero. It records publicly stated emissions and net-zero commitments from countries, sub-national governments, and the world's ~2,000 largest companies.

**Why we need NZT:** SBTi adoption among the US power sector is sparse (only 2 of our 50 cohort parents have SBTi targets — Vistra and NRG). But many cohort parents (Berkshire/MidAmerican, Duke, AEP, Xcel, TVA, etc.) have publicly stated climate commitments outside SBTi. NZT captures these. We use NZT plus 10-K text in Notebook 05 to score parents on the Pledge Quality Score's five pillars.

**Coverage we expect to recover:** roughly 15–30 additional cohort parents (vs the 2 we get from SBTi alone), depending on whether smaller cooperatives are listed.

**Source:** the NZT CSV published at https://zerotracker.net/data. If the auto-download URL has moved, the cell prints `[ACTION REQUIRED]` with manual-download instructions.

In [29]:
NZT_DIR = DATA_RAW / "nzt"
NZT_DIR.mkdir(parents=True, exist_ok=True)
NZT_PROC = DATA_PROCESSED / "nzt_us_companies.csv"

# Locate ANY NZT file in the directory (xlsx preferred since NZT distributes that natively)
def find_nzt_file(d):
    xls = sorted(d.glob("*.xlsx"))
    csvs = sorted(d.glob("*.csv"))
    return xls[0] if xls else (csvs[0] if csvs else None)

nzt_file = find_nzt_file(NZT_DIR)

if nzt_file is None:
    # Try programmatic download first
    NZT_URL_CANDIDATES = [
        "https://zerotracker.net/assets/data/Companies.csv",
        "https://zerotracker.net/data/companies.csv",
    ]
    target = NZT_DIR / "nzt_companies.csv"
    for url in NZT_URL_CANDIDATES:
        try:
            http_download(url, target, headers={"User-Agent": "Mozilla/5.0 (research project)"})
            if target.exists() and target.stat().st_size > 1000:
                nzt_file = target
                break
        except Exception as e:
            print(f"  [miss] {url}: {e}")
    if nzt_file is None:
        print()
        print("[ACTION REQUIRED] could not auto-download Net Zero Tracker.")
        print("  1. Visit https://zerotracker.net/data")
        print("  2. Download the 'Current Snapshot' (xlsx) or Companies CSV")
        print(f"  3. Save anywhere inside {NZT_DIR} (any filename works; .xlsx or .csv)")
        print("  4. Re-run this cell.")
else:
    print(f"[present] NZT file: {nzt_file.name} ({nzt_file.stat().st_size:,} bytes)")


# If we have a file, process and save the canonical CSV for Notebook 05
if nzt_file is not None:
    if nzt_file.suffix.lower() == ".xlsx":
        xl = pd.ExcelFile(nzt_file)
        # NZT distributes a multi-sheet workbook; the data sheet is "Current Snapshot" with header on row 2 (1-indexed) i.e. header=1
        sheet = "Current Snapshot" if "Current Snapshot" in xl.sheet_names else xl.sheet_names[0]
        nzt_raw = pd.read_excel(nzt_file, sheet_name=sheet, header=1)
        # Also save the metadata glossary if present
        if "Metadata Glossary" in xl.sheet_names:
            meta = pd.read_excel(nzt_file, sheet_name="Metadata Glossary")
            meta.to_csv(DATA_PROCESSED / "nzt_metadata_glossary.csv", index=False)
            print(f"  saved metadata glossary: data/processed/nzt_metadata_glossary.csv ({len(meta)} rows)")
    else:
        nzt_raw = pd.read_csv(nzt_file, low_memory=False)

    print(f"\nNZT total rows: {len(nzt_raw):,}, columns: {len(nzt_raw.columns)}")
    if "Entity_type" in nzt_raw.columns:
        print(f"Entity_type counts: {nzt_raw['Entity_type'].value_counts().to_dict()}")

    # Filter to US companies for downstream use
    type_col = next((c for c in nzt_raw.columns if c.lower() == "entity_type"), None)
    country_col = next((c for c in nzt_raw.columns if c.lower() == "country" or c.lower() == "location"), None)
    if type_col and country_col:
        us_co = nzt_raw[(nzt_raw[type_col].astype(str).str.contains("Company|Corporate", case=False, na=False))
                         & (nzt_raw[country_col].astype(str).str.contains("United States|USA|US$", case=False, na=False, regex=True))].copy()
    else:
        us_co = nzt_raw

    us_co.to_csv(NZT_PROC, index=False)
    print(f"Saved {NZT_PROC.name}: {len(us_co):,} US companies × {len(us_co.columns)} cols")
    print(f"  (downstream Notebook 05 reads from this processed CSV, not the raw xlsx)")

[present] NZT file: current_snapshot_2026-05-22_06-40-04.xlsx (3,588,828 bytes)
  saved metadata glossary: data/processed/nzt_metadata_glossary.csv (71 rows)

NZT total rows: 4,190, columns: 66
Entity_type counts: {'Company': 2086, 'City': 1185, 'Region': 721, 'Country': 198}
Saved nzt_us_companies.csv: 686 US companies × 66 cols
  (downstream Notebook 05 reads from this processed CSV, not the raw xlsx)


## 8. Data manifest

A single JSON file at `data/processed/data_manifest.json` recording, for every file in `data/raw/` and `data/processed/`: relative path, size in bytes, SHA-256, and last-modified timestamp.

This manifest is the reproducibility anchor for the project: a future reviewer cloning the repo, re-running this notebook, and comparing their manifest against ours can verify their downloaded files match ours bit-for-bit. Downstream notebooks (02–06) can validate their inputs against the manifest before running expensive analyses.

In [23]:
def make_manifest(roots: list[Path]) -> list[dict]:
    rows = []
    for root in roots:
        for p in sorted(root.rglob("*")):
            if p.is_file() and not p.name.startswith("."):
                rows.append({
                    "path": str(p.relative_to(PROJECT_ROOT)),
                    "size_bytes": p.stat().st_size,
                    "sha256": sha256_of(p),
                    "mtime_utc": datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc)
                                          .isoformat(),
                })
    return rows

manifest_rows = make_manifest([DATA_RAW, DATA_PROCESSED])
manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "n_files": len(manifest_rows),
    "files": manifest_rows,
}
manifest_path = DATA_PROCESSED / "data_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Wrote: {manifest_path}")
print(f"Files indexed: {manifest['n_files']:,}")
pd.DataFrame(manifest_rows).tail(10)

Wrote: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/data/processed/data_manifest.json
Files indexed: 256


,path,size_bytes,sha256,mtime_utc
246,data/raw/sec_10k/THE SOUTHERN CO/2026-02-19__s...,15639198,46aea07b6bd44b32b75014da650ae2610949a487927dd5...,2026-05-22T23:21:21.324960+00:00
247,data/raw/sec_10k/UNS ENERGY CORP/2026-03-13__u...,2109657,3d34b0edd9ace574a084dc503bdb2010f264d77d8600c5...,2026-05-22T22:09:35.412636+00:00
248,data/raw/sec_10k/Vistra Corp/2026-02-27__vistr...,5587052,7f14fefedd12aa959de58c3ab44de4c2eb77d12ec1c4e9...,2026-05-22T21:16:26.005322+00:00
249,data/raw/sec_10k/WEC Energy Group Inc/2026-02-...,6590988,83ad5b5472981b4c9ce47ab2e5a070f0fcbecb1a6d387b...,2026-05-22T21:16:33.989547+00:00
250,data/raw/sec_10k/XCEL ENERGY INC/2026-02-25__x...,5225223,0f5acaa16e280c22d3e755f6c2061683e136ea6dc45e41...,2026-05-22T21:16:27.757916+00:00
251,data/processed/cohort_top50.csv,2329,95b0a12b8e2b8ea6702edc2981d28cac078688f1bc51ce...,2026-05-23T00:45:19.006840+00:00
252,data/processed/data_manifest.json,77372,ccd01c2cdf6cc0724f5eeb0199e6eec1364b71b2299795...,2026-05-23T00:24:04.131115+00:00
253,data/processed/parent_to_cik.csv,2172,6bb3c9800d3e7788ae792e2befb6b9441dfdf146db892b...,2026-05-23T00:45:21.084149+00:00
254,data/processed/power_facility_parent_attributi...,198621,7c407ffc4e169f16fcdabf8ffae6e605f6747b48b41605...,2026-05-23T00:45:19.021695+00:00
255,data/processed/tenk_filings_index.csv,4366,c906f5b9d5fbcdd67c709322f5e787449f09b09de34704...,2026-05-23T00:45:36.114406+00:00


---

## Next notebook

**`01_EDA_GHGRP.ipynb`** — exploratory data analysis on the 13-year GHGRP panel restricted to the locked top-50 cohort. Eight sections:

1. **Panel construction** — unpivot the wide-form GHGRP workbook to long-form facility-year, attach 2023 ownership, restrict to cohort. Produces `cohort_facility_year_panel.csv` and `cohort_parent_year_panel.csv` — the inputs to every subsequent notebook.
2. **Cohort trajectory overview** — headline chart for the working paper.
3. **Fuel-mix composition** — coal / gas / oil / biomass decomposition from the unit-and-fuel workbook.
4. **Per-parent trajectory regression** — OLS slope + bootstrap CI, leaders/laggards table.
5. **Threshold-effect handling** — hand-labeled exit register for the top-20 dropouts (retirement / sub-threshold / rule-change).
6. **Geographic concentration** — state-level rollup.
7. **CEMS sanity peek** — visual side-by-side for top-10 parents (the proper LME comparison is Notebook 03).
8. **Save deliverables** — canonical panel CSVs to `data/processed/`, figures to `outputs/figures/`.

See `workplan.md` for the full notebook-by-notebook plan and current execution status.